### Setup and Imports

In [1]:
# ── Import everything we'll need for this class ──────────────────────────
import requests                          # HTTP fetching
from bs4 import BeautifulSoup            # HTML parsing
import pandas as pd                      # Tabular data
import sqlite3                           # Local database
import time                              # Polite pauses between requests
import os
from sqlalchemy import create_engine, text

print('✅ All libraries imported successfully')

✅ All libraries imported successfully


### Srape 10 pages 

In [ ]:
# testing the http request

res = requests.get('https://books.toscrape.com/catalogue/page-1.html')

print(res.status_code)

200


In [ ]:
# creat a safe_get function that returns the page html 

def safe_get(url, delay=1): 
    """Fetch a URL, return None if it fails, and sleep politely after."""
    try:
        response = requests.get(url, timeout=10) 
        response.raise_for_status()    # Raises exception for 4xx / 5xx codes
        time.sleep(delay)              
        return response
    except requests.RequestException as e:
        print(f'Request failed: {e}')
        return None

resp = safe_get('https://books.toscrape.com')
if resp:
    print(f'✅ Got {len(resp.text):,} chars')


✅ Got 51,294 chars


In [ ]:
# Create a for loop to scrape 10 pages from books.toscrape.com
import datetime


url = 'https://books.toscrape.com/catalogue/page-{}.html'
all_books = []

for page_num in range(1, 11):  # loop through the pages
    resp = safe_get(url.format(page_num)) # get the response from the page

    if resp is None: 
        print(f'⚠️ Skipping page {page_num} — request failed')
        continue

    soup = BeautifulSoup(resp.text, 'html.parser')

    for book in soup.find_all('article', class_='product_pod'): # find all the books in the page
        title  = book.find('h3').find('a')['title'] # find the title of the book
        price  = book.find('p', class_='price_color').get_text(strip=True) # find the price of the book
        rating = book.find('p', class_='star-rating')['class'][1] # find the rating of the book
        avail  = book.find('p', class_='instock').get_text(strip=True) # find the availability of the book

        all_books.append({  # append the book to the list of books
            'title'       : title,
            'price'       : price,
            'rating'      : rating,
            'availability': avail,
            'page'        : page_num
        })



In [ ]:
#create a dataframe from the list of books and print the first 5 rows
books_df = pd.DataFrame(all_books)  # create a dataframe from the list of books

print(books_df.shape) # print the shape of the dataframe

books_df.head(5) # print the first 5 rows of the dataframe

(200, 5)


,title,price,rating,availability,page
0,A Light in the Attic,Â£51.77,Three,In stock,1
1,Tipping the Velvet,Â£53.74,One,In stock,1
2,Soumission,Â£50.10,One,In stock,1
3,Sharp Objects,Â£47.82,Four,In stock,1
4,Sapiens: A Brief History of Humankind,Â£54.23,Five,In stock,1


### Clean and Transform

In [ ]:
# pass the books_df variable to a new variable
books_df_new = books_df

print(books_df_new.duplicated().sum()) #check for duplicates
print(books_df_new.info()) # check the data information
print(books_df_new.dtypes) # check the datatypes


np.int64(0)

In [ ]:

#Remove the pounds symbol from price column
books_df_new['price'] = books_df_new['price'].str.replace('Â£', '').strip()


In [45]:
# Add scraped_at column to the dataframe
books_df_new['scraped_at'] = pd.Timestamp.now().replace(second=0, microsecond=0)


In [46]:
# convert the rating columns from words to numbers
rating_map = {
    'One': 1,
    'Two': 2,
    'Three': 3,
    'Four': 4,
    'Five': 5,
}

books_df_new['rating'] = books_df_new['rating'].map(rating_map)

In [ ]:
# convert all columns to the right datatypes 

books_df_new = books_df_new.astype({
    'title': 'str',
    'price': 'float',
    'rating': 'Int64',
    'availability': 'str',
    'page': 'Int64',
    'scraped_at': 'datetime64[ns]'
})

In [53]:
books_df_new.dtypes

title                   object
price                  float64
rating                   Int64
availability            object
page                     Int64
scraped_at      datetime64[ns]
dtype: object